# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yowxy/Flyrank-Internship-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**My lane: Content Refresh — "out of thousands of pages, which page should an editor fix FIRST?"**
(FlyRank builds content as infrastructure, then watches search data and optimises. Pages rank, then quietly decay. The valuable decision is the refresh queue order.)

**One-paragraph frame:** For a content editor deciding which pages to refresh first, we will build a ranked queue from trailing-90-day search + engagement + content signals, predicting a per-page decline flag measured by ROC-AUC and precision@200. A wrong call costs wasted editor hours (false alarm refreshed for nothing) or a missed decline (traffic keeps bleeding). A plain rule isn't enough because decline comes from many tangled, shifting signals no single threshold separates. We will claim only observed, measured, decision-support results.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Answer: binary classification that is USED as scoring/ranking.**

- The core learnable question is **"will this page decline?"** — a yes/no label per page (`is_declining_label`). That is **classification** (task-type table: "will this one decline/recover?" → classification, target = yes/no label from an observed outcome).
- At decision time the editor does not act on one yes/no in isolation — they work down a **ranked queue** ("which ones first?" → ranking/scoring). So the classifier's predicted probability becomes a **priority score**, and pages are sorted by it.
- It is **not clustering**: I am not asking "what kinds of pages exist?" with no target — I have a concrete outcome (decline) and an action (refresh order).
- It is **not pure signal analysis**: I need a per-page decision, not just "which signals travel together."

So: **train a classifier, ship a ranked list.** Success is judged both ways (Section 3): classifier quality (ROC-AUC) AND queue quality (precision@K).

In [1]:
# §1 — CODE: what does the classification setup look like? (base rate = what "dumb" gets you)
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv" if __import__("os").path.exists("data/raw/content_refresh_anonymized.csv") else "../../data/raw/content_refresh_anonymized.csv")
# Rebuild the target exactly as the pipeline defines it (NOT a feature):
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"rows: {len(df):,} | clients: {df['client_id'].nunique()} | one row = one content item (page)")
print(f"declining (label=1): {df['is_declining_label'].sum():,} | base rate: {base_rate:.3f}")
print("\nA majority-class dummy scores accuracy = %.3f and ROC-AUC = 0.50. Anything we build must beat that." % max(base_rate, 1-base_rate))

rows: 30,000 | clients: 32 | one row = one content item (page)
declining (label=1): 16,262 | base rate: 0.542

A majority-class dummy scores accuracy = 0.542 and ROC-AUC = 0.50. Anything we build must beat that.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label` = 1 when `trend_direction == "down"`, else 0.**

- **Where it comes from:** `trend_direction` compares **observed** impressions in the most recent 30 days vs the 30 days before that: `down` means `impressions_last_30d` is more than 20% below `impressions_prev_30d` (with `new`/`flat` edge cases when the previous window is zero). The label is therefore a **thresholded rule applied to an observed outcome** — a proxy for "this page needs a refresh."
- **Observed vs defined (honest version):** the underlying quantities (impressions in two windows) are measured behaviour from Google Search Console, not someone's opinion — good. But the −20% cutoff itself is a human-chosen line, so the label is a **proxy**, not a law of nature. A page at −19% and one at −21% are nearly identical but get different labels. I accept this because the business needs a crisp queue, and I mitigate it by also evaluating the underlying **score ranking** (precision@K), not just the 0/1 line.
- **Leakage discipline:** `trend_pct` and `trend_direction` COMPUTE the label, so they (and `is_declining_label` itself) are **never features**. Features may only use information available *before* the outcome window — the classic past→future split the later notebooks enforce.
- **Alternatives I considered and rejected:** predicting raw `trend_pct` (regression) is noisier and harder for an editor to act on; predicting clicks/sessions mixes demand size with decay (a huge page can lose 30% and still dwarf a small page). The binary decline flag maps directly onto the refresh action.

In [2]:
# §2 — CODE: label construction + leakage-column quarantine check
print("trend_direction distribution (label source):")
print(df["trend_direction"].value_counts())
print("\nis_declining_label distribution:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))
# Confirm the definition holds exactly:
assert ((df["trend_direction"] == "down").astype(int) == df["is_declining_label"]).all()
print("\nOK: is_declining_label == (trend_direction == 'down') on all 30,000 rows.")
print("LEAKAGE RULE: trend_pct, trend_direction are label machinery -> must NEVER be features.")
print("Blank trend_pct (prev_30d == 0):", int(df["trend_pct"].isna().sum()), "-> prep fills with 0; 'new'/'flat' rows land in label=0.")

trend_direction distribution (label source):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label distribution:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64

OK: is_declining_label == (trend_direction == 'down') on all 30,000 rows.
LEAKAGE RULE: trend_pct, trend_direction are label machinery -> must NEVER be features.
Blank trend_pct (prev_30d == 0): 3388 -> prep fills with 0; 'new'/'flat' rows land in label=0.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary: ROC-AUC on client-holdout pages. Decision metric alongside it: precision@200.**

- **Why ROC-AUC first:** it measures ranking quality across all thresholds (does a declining page score above a stable one?), which is exactly the queue-ordering skill. It is insensitive to the 54.2% base rate, so it stays comparable as the mix shifts across clients. Random = 0.50.
- **Why precision@200 second:** an editor can only fix a handful of pages per sprint. If we hand them the top-200 scores, what fraction truly are declining? Random ordering gives ~0.54 (the base rate); a useful queue must clearly beat that.
- **What number means "good"?** I name it BEFORE training: **ROC-AUC ≥ 0.70 on held-out clients** (clearly above random, honest for noisy search data — not a 0.95 fantasy), **and precision@200 ≥ 0.65** (at least ~11 points above the 0.54 base rate, i.e. ~130 of the top 200 are real declines). Both must hold on **client-holdout** splits (never-shown clients), because per-client history depth differs wildly and a within-client split would flatter us.
- **Error costs:** a false positive wastes editor hours refreshing a healthy page; a false negative lets a decay keep bleeding. Because editor time is the scarcer resource, I weight the queue's precision over raw recall — hence precision@K next to AUC rather than recall alone.

In [3]:
# §3 — CODE: what "good" must beat — random baselines computed on THIS data
import numpy as np
base = df["is_declining_label"].mean()
print(f"base rate (declining share): {base:.3f}")
print("Random-queue precision@K expectation ~= base rate =", round(base, 3), "at ANY K.")
print("Majority-class dummy: accuracy = %.3f, ROC-AUC = 0.500" % max(base, 1-base))
K = 200
expected_hits_random = K * base
print(f"\nAt K={K}: a random top-{K} contains ~{expected_hits_random:.0f} real declines ({base:.1%}).")
print(f"My bar  : ROC-AUC >= 0.70 AND precision@{K} >= 0.65 (i.e. >={int(K*0.65)} real declines in top-{K}).")
print("Splits for honesty: group by client_id (32 clients) -> client-holdout, never row-shuffle.")

base rate (declining share): 0.542
Random-queue precision@K expectation ~= base rate = 0.542 at ANY K.
Majority-class dummy: accuracy = 0.542, ROC-AUC = 0.500

At K=200: a random top-200 contains ~108 real declines (54.2%).
My bar  : ROC-AUC >= 0.70 AND precision@200 >= 0.65 (i.e. >=130 real declines in top-200).
Splits for honesty: group by client_id (32 clients) -> client-holdout, never row-shuffle.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page), aggregated over a trailing 90-day window ending at export time.** 30,000 rows, 32 pseudonymised clients. Grain check below confirms `content_id` is unique (no duplicates). IDs are pseudonyms: grouping/splitting only, never features. Rate columns are ×100 percentages (`ctr = 0.76` means 0.76%). `avg_position = 0` means "no position data" (not rank zero). The last column sketches the target: `is_declining_label` rebuilt from `trend_direction`.

In [4]:
# §4 — CODE: the unit of analysis as a REAL dataframe (+ sketched target column)
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv" if __import__("os").path.exists("data/raw/content_refresh_anonymized.csv") else "../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)  # sketched target
print(f"source shape: {df.shape} (44 source cols + 1 sketched target) | clients: {df['client_id'].nunique()} | content_id unique: {df['content_id'].is_unique}")
show = df[["content_type","main_intent","impressions_90d","clicks_90d","sessions_90d",
           "avg_position","ctr","content_age_days","days_since_last_update",
           "impressions_last_30d","impressions_prev_30d","is_declining_label"]].head(8)
print("\nOne row = one page (trailing-90-day snapshot) + sketched target column:")
print(show.to_string(index=False))
print("\nNumeric snapshot (note ×100 rate scale; avg_position 0 = no data):")
print(df[["impressions_90d","clicks_90d","sessions_90d","avg_position","ctr","content_age_days"]].describe().round(2).to_string())
print("\nMissingness follows content_type (systematic, not random) — never blind fillna(0):")
print(pd.crosstab(df["content_type"], df["search_volume"].isna(), rownames=["content_type"], colnames=["search_volume_missing"]).to_string())
print("\nTarget sketch: declining =", int(df["is_declining_label"].sum()), f"({df['is_declining_label'].mean():.1%}); stable/up/new/flat = 0.")

source shape: (30000, 45) (44 source cols + 1 sketched target) | clients: 32 | content_id unique: True

One row = one page (trailing-90-day snapshot) + sketched target column:
   content_type   main_intent  impressions_90d  clicks_90d  sessions_90d  avg_position  ctr  content_age_days  days_since_last_update  impressions_last_30d  impressions_prev_30d  is_declining_label
keyword article transactional             3803          29            17          10.6 0.76               187                      20                   578                   987                   1
keyword article informational            15320           7             9          20.3 0.05               445                      25                  2501                  5915                   1
keyword article informational            12581          11            11          36.5 0.09               141                      20                  2382                  6089                   1
keyword article    commercial   

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

FlyRank already flags pages with hand-written rules (a health score, if-this-then-that thresholds). They run in production and they work — up to a point. Three reasons a fixed rule runs out here, each checked in the code below:

1. **No single threshold separates decline.** Declining and healthy pages overlap heavily on every obvious cutoff (position, age, traffic level). Any one-line rule like "refresh if position > 20" either drowns the editor in false alarms or misses most real declines — the code measures exactly that.
2. **The signals are many and tangled.** Decline lives in the *combination* of demand (impressions, keyword competition), rank (position, CTR), engagement (sessions, engagement/scroll rates), freshness (age, days since update), and content type — with systematic missingness along content-type lines. Hand-tuning a 10-way interaction is guesswork; a model learns the weighting from data.
3. **It shifts across clients.** 32 clients with different niches, volumes, and history depths: a threshold tuned on one client misfires on the next. A model evaluated on **held-out clients** learns portable patterns instead of one site's magic number — and it re-learns as data drifts, where a frozen rule rots.

The honest division of labour: keep the rules as the **baseline to beat**, never as features (that would be circular — the model would just memorise the flag). ML earns its place only if its ranked queue beats the best rule's queue on precision@K — tested in notebook 04.

In [5]:
# §5 — CODE: a best-effort single rule fails -> the pattern needs combining
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv" if __import__("os").path.exists("data/raw/content_refresh_anonymized.csv") else "../../data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"]
# Candidate rule: "stale AND badly ranked" (generous hand-picked thresholds)
rule = ((df["avg_position"] > 20) & (df["days_since_last_update"] > 180)).fillna(False)
tp = int((rule & (y == 1)).sum()); fp = int((rule & (y == 0)).sum())
prec = tp / max(len(df[rule]), 1); rec = tp / max(int(y.sum()), 1)
print(f"Rule 'position>20 & stale>180d' flags {int(rule.sum()):,} pages ({rule.mean():.1%})")
print(f"  precision = {prec:.3f} (vs base rate {y.mean():.3f}) | recall = {rec:.3f} (catches {tp:,}/{int(y.sum()):,} declines)")
print("  -> barely above base rate and misses >99% of declines: one threshold can't do the job.")
print("\nOverlap proof (medians by label — note how close they sit):")
print(df.groupby("is_declining_label")[["avg_position","content_age_days","days_since_last_update","impressions_90d","ctr"]].median().round(1).to_string())
print("\nMany tangled signals + overlap + per-client shift = learned weighting beats hand thresholds.")

Rule 'position>20 & stale>180d' flags 26 pages (0.1%)
  precision = 0.538 (vs base rate 0.542) | recall = 0.001 (catches 14/16,262 declines)
  -> barely above base rate and misses >99% of declines: one threshold can't do the job.

Overlap proof (medians by label — note how close they sit):
                    avg_position  content_age_days  days_since_last_update  impressions_90d  ctr
is_declining_label                                                                              
0                           10.0             287.0                    20.0            472.0  0.0
1                           11.3             216.0                    20.0            961.0  0.1

Many tangled signals + overlap + per-client shift = learned weighting beats hand thresholds.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Claim discipline used:** observed (measured impression windows), proxy (the −20% line is defined), directional / decision-support (a refresh queue to assist editors — never "we predicted Google's algorithm").